# core

> Utilities for managing SLURM jobs, SSH connections, and port forwarding on HPC clusters.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_ne, test_fail

In [ ]:
#| export
import socket

def find_free_port(above=8000):
    "Find a free port above `above`"
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        port = s.getsockname()[1]
        while port <= above:
            s.close()
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.bind(('', 0))
            port = s.getsockname()[1]
        return port

In [ ]:
port = find_free_port()
assert port > 8000, f"Expected port > 8000, got {port}"
assert port < 65536
print(f"Found free port: {port}")

Found free port: 59057


In [ ]:
#| export
import subprocess

def start_or_connect(job_name, host, command=None,
                     slurm_args="--qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00"):
    """SSH to host and manage a SLURM job with nested tmux sessions.

    Checks (via ssh+squeue) whether a SLURM job named `job_name` is already running.
    - If running: reattaches to the existing outer tmux session on the login node.
    - If not running: creates an outer tmux on the login node that runs
      `salloc` to get a compute node, then starts an inner tmux on the compute
      node via `srun --pty`. Optionally runs `command` inside the inner tmux.

    params:
        job_name: name for both the SLURM job and tmux sessions
        host: SSH host alias for the login node (must be in ~/.ssh/config)
        command: optional command to run in the inner tmux's initial window
        slurm_args: arguments passed to salloc (resources, time, qos, etc.)
    """
    # Check if job is already running
    check = subprocess.run(
        ["ssh", host, f"squeue --me --name={job_name} --states=RUNNING --noheader"],
        capture_output=True, text=True
    )
    job_running = bool(check.stdout.strip())

    if job_running:
        print(f"Job '{job_name}' is already running. Reattaching to outer tmux...")
        ssh_cmd = f'ssh -t {host} "tmux new-session -A -s {job_name}"'
    else:
        print(f"No running job '{job_name}' found. Starting salloc + inner tmux...")
        # Inner tmux runs on compute node via srun
        inner_tmux = f"tmux new-session -s {job_name}"
        if command:
            # Run command in the inner tmux window, then exec bash so the window stays alive
            inner_tmux += f" \\\"{command}; exec bash\\\""

        salloc_cmd = f"salloc --job-name={job_name} {slurm_args} srun --pty {inner_tmux}"
        # Outer tmux on login node runs salloc as its shell command
        ssh_cmd = f"ssh -t {host} \"tmux new-session -A -s {job_name} '{salloc_cmd}'\""

    print(ssh_cmd)
    return ssh_cmd

In [ ]:
from unittest.mock import patch, MagicMock

def _mock_check(stdout=""):
    """Create a mock subprocess.run result with given stdout."""
    mock_result = MagicMock()
    mock_result.stdout = stdout
    mock_result.stderr = ""
    return mock_result

# Case 1: Job IS running → just reattach to outer tmux
with patch("subprocess.run", return_value=_mock_check("g3098")):
    cmd = start_or_connect("proxy_jump", "klone-login")
test_eq(cmd, 'ssh -t klone-login "tmux new-session -A -s proxy_jump"')

print("---")

# Case 2: Job NOT running, no command → salloc + srun + inner tmux
with patch("subprocess.run", return_value=_mock_check("")):
    cmd = start_or_connect("proxy_jump", "klone-login", slurm_args="--time=01:00:00")
test_eq(cmd, 'ssh -t klone-login "tmux new-session -A -s proxy_jump \'salloc --job-name=proxy_jump --time=01:00:00 srun --pty tmux new-session -s proxy_jump\'"')

print("---")

# Case 3: Job NOT running, with command → salloc + srun + inner tmux + command
with patch("subprocess.run", return_value=_mock_check("")):
    cmd = start_or_connect("proxy_jump", "klone-login", command="nvidia-smi", slurm_args="--time=01:00:00")
test_eq(cmd, 'ssh -t klone-login "tmux new-session -A -s proxy_jump \'salloc --job-name=proxy_jump --time=01:00:00 srun --pty tmux new-session -s proxy_jump \\\"nvidia-smi; exec bash\\\"\'"')

Job 'proxy_jump' is already running. Reattaching to outer tmux...
ssh -t klone-login "tmux new-session -A -s proxy_jump"
---
No running job 'proxy_jump' found. Starting salloc + inner tmux...
ssh -t klone-login "tmux new-session -A -s proxy_jump 'salloc --job-name=proxy_jump --time=01:00:00 srun --pty tmux new-session -s proxy_jump'"
---
No running job 'proxy_jump' found. Starting salloc + inner tmux...
ssh -t klone-login "tmux new-session -A -s proxy_jump 'salloc --job-name=proxy_jump --time=01:00:00 srun --pty tmux new-session -s proxy_jump \"nvidia-smi; exec bash\"'"


In [ ]:
#| export
def get_job_node(job_name, host):
    """Get the node that is running the job with name `job_name` on host `host`.
    Prints and returns the ssh command that queries squeue for the node.
    """
    squeue_cmd = (
        f"squeue --me --name={job_name} --states=RUNNING "
        f"--Format=NodeList --noheader"
    )
    ssh_cmd = f"ssh {host} '{squeue_cmd}'"
    print(ssh_cmd)
    return ssh_cmd

In [ ]:
cmd = get_job_node("proxy_jump", "klone-login")
test_eq(cmd, "ssh klone-login 'squeue --me --name=proxy_jump --states=RUNNING --Format=NodeList --noheader'")

ssh klone-login 'squeue --me --name=proxy_jump --states=RUNNING --Format=NodeList --noheader'


In [ ]:
#| export
def get_port_forwarding_command(local_port, remote_port, host, node, local_network="hyak.local"):
    """Get the command to forward local_port to remote_port on a compute node via the login host.
    Uses SSH local port forwarding with a ProxyJump through the login node.
    This function would not run the actual command, just print it to screen.
    """
    cmd = (
        f"ssh -N -f "
        f"-L {local_port}:{node}.{local_network}:{remote_port} "
        f"{host}"
    )
    print(cmd)
    return cmd

In [ ]:
cmd = get_port_forwarding_command(8080, 8000, "klone-login", "g3098")
test_eq(cmd, "ssh -N -f -L 8080:g3098.hyak.local:8000 klone-login")

# Custom network
cmd2 = get_port_forwarding_command(9090, 9000, "tillicum-login", "n1234", local_network="internal.net")
test_eq(cmd2, "ssh -N -f -L 9090:n1234.internal.net:9000 tillicum-login")

ssh -N -f -L 8080:g3098.hyak.local:8000 klone-login
ssh -N -f -L 9090:n1234.internal.net:9000 tillicum-login


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()